## EDA - Exploratory Data Analysis
Dataset Link: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [4]:
## loading dataset
data = pd.read_csv(r"C:\Users\himan\Desktop\Projects\pathwayIQ\dataset\raw\diabetic_data.csv")
data.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
# converting to dataframe 
df = pd.DataFrame(data)
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [6]:
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [7]:
df["readmitted"].value_counts(normalize=True)

readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

In [8]:
## Class is higly imbalanced. Let's balance it
df["target"] = df["readmitted"].apply(lambda x: 1 if x == "<30" else 0)
df["target"].value_counts(normalize=True)

target
0    0.888401
1    0.111599
Name: proportion, dtype: float64

In [9]:
# Dropping 3 columns as they are now redundant
df.drop(columns=["encounter_id", "patient_nbr", "readmitted"], inplace=True)

In [10]:
(df == "?").sum().sort_values(ascending=False).head(10)

weight                 98569
medical_specialty      49949
payer_code             40256
race                    2273
diag_3                  1423
diag_2                   358
diag_1                    21
admission_type_id          0
time_in_hospital           0
admission_source_id        0
dtype: int64

In [11]:
df.drop(columns=["weight"], inplace=True)

In [12]:
df["medical_specialty"].nunique()

73

In [13]:
df["medical_specialty"].value_counts().head(15)

medical_specialty
?                                  49949
InternalMedicine                   14635
Emergency/Trauma                    7565
Family/GeneralPractice              7440
Cardiology                          5352
Surgery-General                     3099
Nephrology                          1613
Orthopedics                         1400
Orthopedics-Reconstructive          1233
Radiologist                         1140
Pulmonology                          871
Psychiatry                           854
Urology                              685
ObstetricsandGynecology              671
Surgery-Cardiovascular/Thoracic      652
Name: count, dtype: int64

In [14]:
top_specialties = df["medical_specialty"].value_counts().nlargest(10).index

df["medical_specialty"] = df["medical_specialty"].apply(
    lambda x: x if x in top_specialties else "Other"
)

In [15]:
df["medical_specialty"].value_counts()

medical_specialty
?                             49949
InternalMedicine              14635
Other                          8340
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Name: count, dtype: int64

In [16]:
df["medical_specialty"] = df["medical_specialty"].replace("?", np.nan)

In [17]:
df["medical_specialty"].isnull().sum()

np.int64(49949)

In [18]:
df.groupby(df["medical_specialty"].isnull())["target"].mean()

medical_specialty
False    0.107609
True     0.115738
Name: target, dtype: float64

Missingness has a slightly higher readmission rate:

non-missing → ~10.76%,  
missing → ~11.57%

In [19]:
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")

In [20]:
df["payer_code"].value_counts().head(15)

payer_code
?     40256
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2533
UN     2448
CM     1937
OG     1033
PO      592
DM      549
CH      146
WC      135
OT       95
Name: count, dtype: int64

In [21]:
df["payer_code"].nunique()

18

In [22]:
df["payer_code"] = df["payer_code"].replace("?", "Unknown")

In [24]:
df["payer_code"].value_counts().head(10)

payer_code
Unknown    40256
MC         32439
HM          6274
SP          5007
BC          4655
MD          3532
CP          2533
UN          2448
CM          1937
OG          1033
Name: count, dtype: int64

In [25]:
df["race"].value_counts(dropna=False)

race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

In [26]:
df["race"] = df["race"].replace("?", "Unknown")

In [27]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
numerical_cols = df.select_dtypes(exclude="object").columns.tolist()

print("Categorical:", categorical_cols)
print("\nNumerical:", numerical_cols)

Categorical: ['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']

Numerical: ['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'target']


In [28]:
id_like_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in id_like_cols:
    df[col] = df[col].astype(str)

In [29]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
numerical_cols = df.select_dtypes(exclude="object").columns.tolist()

print("Categorical:", len(categorical_cols))
print("Numerical:", numerical_cols)

Categorical: 38
Numerical: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'target']


In [30]:
df["discharge_disposition_id"].value_counts().head(15)

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
Name: count, dtype: int64

In [31]:
df.groupby("discharge_disposition_id")["target"].mean().sort_values(ascending=False).head(15)

discharge_disposition_id
12    0.666667
15    0.444444
9     0.428571
28    0.366906
22    0.276969
5     0.208615
2     0.160714
3     0.146625
24    0.145833
7     0.144462
8     0.138889
4     0.127607
6     0.126957
18    0.124357
25    0.093023
Name: target, dtype: float64

In [32]:
df["discharge_disposition_id"].value_counts()

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
8       108
15       63
24       48
9        21
17       14
16       11
19        8
10        6
27        5
12        3
20        2
Name: count, dtype: int64

In [33]:
rare_discharge = df["discharge_disposition_id"].value_counts()

rare_discharge = rare_discharge[rare_discharge < 100].index

df["discharge_disposition_id"] = df["discharge_disposition_id"].apply(
    lambda x: "Other" if x in rare_discharge else x
)

In [34]:
## inspect "Age" column
df["age"].value_counts()

age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

In [35]:
age_map = {
    "[0-10)": 0,
    "[10-20)": 1,
    "[20-30)": 2,
    "[30-40)": 3,
    "[40-50)": 4,
    "[50-60)": 5,
    "[60-70)": 6,
    "[70-80)": 7,
    "[80-90)": 8,
    "[90-100)": 9
}

df["age"] = df["age"].map(age_map)

In [36]:
df.groupby("age")["target"].mean()

age
0    0.018634
1    0.057887
2    0.142426
3    0.112318
4    0.106040
5    0.096662
6    0.111284
7    0.117731
8    0.120835
9    0.110992
Name: target, dtype: float64

In [37]:
## inspect "Gender" Distribution
df["gender"].value_counts(dropna=False)

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

In [38]:
df.groupby("gender")["target"].mean()

gender
Female             0.112452
Male               0.110615
Unknown/Invalid    0.000000
Name: target, dtype: float64

In [39]:
df = df[df["gender"] != "Unknown/Invalid"]

In [40]:
## Inspecting "Diagnosis" Column 
df[["diag_1", "diag_2", "diag_3"]].head(10)

,diag_1,diag_2,diag_3
0,250.83,?,?
1,276,250.01,255
2,648,250,V27
3,8,250.43,403
4,197,157,250
5,414,411,250
6,414,411,V45
7,428,492,250
8,398,427,38
9,434,198,486


In [41]:
df["diag_1"].nunique(), df["diag_2"].nunique(), df["diag_3"].nunique()

(717, 749, 790)

In [42]:
df["diag_1"].value_counts().head(20)

diag_1
428      6862
414      6580
786      4016
410      3614
486      3508
427      2766
491      2275
715      2151
682      2042
434      2028
780      2019
996      1967
276      1889
38       1688
250.8    1680
599      1595
584      1520
V57      1207
250.6    1183
518      1115
Name: count, dtype: int64

In [43]:
df[["diag_1", "diag_2", "diag_3"]].isnull().sum()

diag_1    0
diag_2    0
diag_3    0
dtype: int64

In [44]:
top_diag1 = df["diag_1"].value_counts().nlargest(20).index

df["diag_1"] = df["diag_1"].apply(
    lambda x: x if x in top_diag1 else "Other"
)

In [45]:
df["diag_1"].value_counts()

diag_1
Other    50058
428       6862
414       6580
786       4016
410       3614
486       3508
427       2766
491       2275
715       2151
682       2042
434       2028
780       2019
996       1967
276       1889
38        1688
250.8     1680
599       1595
584       1520
V57       1207
250.6     1183
518       1115
Name: count, dtype: int64

In [46]:
for col in ["diag_2", "diag_3"]:
    top_codes = df[col].value_counts().nlargest(20).index
    
    df[col] = df[col].apply(
        lambda x: x if x in top_codes else "Other"
    )

In [47]:
df[["diag_2", "diag_3"]].nunique()

diag_2    21
diag_3    21
dtype: int64

In [48]:
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin'
]

for col in med_cols:
    print("\n", col)
    print(df[col].value_counts())


 metformin
metformin
No        81776
Steady    18345
Up         1067
Down        575
Name: count, dtype: int64

 repaglinide
repaglinide
No        100224
Steady      1384
Up           110
Down          45
Name: count, dtype: int64

 nateglinide
nateglinide
No        101060
Steady       668
Up            24
Down          11
Name: count, dtype: int64

 chlorpropamide
chlorpropamide
No        101677
Steady        79
Up             6
Down           1
Name: count, dtype: int64

 glimepiride
glimepiride
No        96572
Steady     4670
Up          327
Down        194
Name: count, dtype: int64

 acetohexamide
acetohexamide
No        101762
Steady         1
Name: count, dtype: int64

 glipizide
glipizide
No        89078
Steady    11355
Up          770
Down        560
Name: count, dtype: int64

 glyburide
glyburide
No        91113
Steady     9274
Up          812
Down        564
Name: count, dtype: int64

 tolbutamide
tolbutamide
No        101740
Steady        23
Name: count, dtype: int64

 piog

In [49]:
for col in med_cols:
    dominant_ratio = df[col].value_counts(normalize=True).iloc[0]
    
    if dominant_ratio > 0.99:
        print(col, dominant_ratio)

nateglinide 0.9930917917121154
chlorpropamide 0.9991548991283669
acetohexamide 0.9999901732456787
tolbutamide 0.9997739846506097
acarbose 0.9969733596690349
miglitol 0.99962658333579
troglitazone 0.9999705197370361
tolazamide 0.9996167565814688
examide 1.0
citoglipton 1.0


In [50]:
drop_cols = [
    "examide",
    "citoglipton",
    "acetohexamide",
    "troglitazone",
    "tolbutamide",
    "tolazamide",
    "miglitol",
    "chlorpropamide"
]

df.drop(columns=drop_cols, inplace=True)

In [51]:
df.shape

(101763, 39)

In [54]:
df.to_csv(r"C:\Users\himan\Desktop\Projects\pathwayIQ\dataset\processed\cleaned_diabetic_data.csv", index=False)

#### Observations
- target highly imbalanced (~11% positive)  
- weight dropped due to extreme missingness  
- diagnosis columns grouped to reduce sparsity  
- several medication columns removed due to near-constant values  
- age converted to ordinal encoding  
- categorical ID columns identified  